# Notebook 08 — Full Universe Pair Scan

**Adaptive Pair Trading | Ayush Arora (MQMS2404)**

---

## Objective

Systematically scan the full **79-stock NIFTY 100 universe** (3,081 candidate pairs)
for statistically and economically valid cointegrated pairs using a rigorous
**five-stage hierarchical filter**:

| Stage | Test | Threshold | Pairs passing |
|-------|------|-----------|---------------|
| 1 | Return correlation | > 0.40 | ~600 |
| 2 | Engle–Granger cointegration | p < 0.05 (raw) | ~150 |
| 3 | Benjamini–Hochberg correction | FDR < 5% | ~50 |
| 4 | Johansen trace test (independent) | 95% critical value | ~30 |
| 5 | Spread quality: Hurst + ADF + half-life | H<0.5, ADF p<0.05, HL 5–90d | ~15 |

The **composite score** ranks surviving pairs by a weighted combination of:
- ADF stationarity strength (30%)
- Hurst exponent (25%) — lower is more mean-reverting
- Half-life optimality (20%) — penalises extremes
- Johansen trace ratio (25%) — strength of cointegrating relationship

Output: `data/top_pairs.csv` — forwarded to NB09 (walk-forward) and NB10 (portfolio).

In [ ]:
import pandas as pd
import numpy as np
import itertools
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.stats.multitest import multipletests
from scipy.stats import pearsonr
from config import (
    PRICES_FILE, CORR_THRESHOLD, COINT_ALPHA, ADF_ALPHA,
    MIN_HALF_LIFE, MAX_HALF_LIFE, TOP_N_PAIRS, TOP_PAIRS_FILE, SECTORS
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded.')

## Step 1: Load Universe & Build Candidate Pairs

We generate pairs **within each sector** (preserves economic logic) plus a
**cross-sector screen** for highly correlated tickers — occasionally, structurally
related companies span sector boundaries (e.g., ONGC and GAIL share gas infrastructure).

In [ ]:
prices = pd.read_csv(PRICES_FILE, index_col=0, parse_dates=True)
print(f'Universe: {prices.shape[1]} stocks  |  {prices.shape[0]} trading days')
print(f'Date range: {prices.index[0].date()} → {prices.index[-1].date()}')

# Reverse-map stock → sector
stock_to_sector = {}
for sector, members in SECTORS.items():
    for s in members:
        if s in prices.columns:
            stock_to_sector[s] = sector

avail = [c for c in prices.columns if c in stock_to_sector]
print(f'\nStocks with sector mapping: {len(avail)} / {prices.shape[1]}')

## Step 2: Correlation Filter

Pairwise return correlation filters out pairs with insufficient co-movement
before the more expensive cointegration tests are run.

We generate within-sector pairs from the sector map **and** cross-sector pairs
from all 79 stocks that pass the correlation threshold — whichever source they come from.

In [ ]:
returns = prices.pct_change().dropna()

# All pairs from stocks with sector mapping
all_candidates = list(itertools.combinations(avail, 2))
print(f'Total candidate pairs (sector-mapped stocks): {len(all_candidates)}')

corr_records = []
for a, b in all_candidates:
    c, _ = pearsonr(returns[a].dropna(), returns[b].dropna())
    if c >= CORR_THRESHOLD:
        same_sector = stock_to_sector.get(a) == stock_to_sector.get(b)
        corr_records.append({
            'Stock A': a, 'Stock B': b,
            'Sector A': stock_to_sector.get(a, 'Unknown'),
            'Sector B': stock_to_sector.get(b, 'Unknown'),
            'Same Sector': same_sector,
            'Correlation': round(c, 4)
        })

corr_df = pd.DataFrame(corr_records).sort_values('Correlation', ascending=False)
print(f'Pairs passing correlation filter (r > {CORR_THRESHOLD}): {len(corr_df)}')
print(f'  Within-sector: {corr_df["Same Sector"].sum()}')
print(f'  Cross-sector : {(~corr_df["Same Sector"]).sum()}')
corr_df.head(10)

## Step 3: Engle–Granger Cointegration + Benjamini–Hochberg Correction

### Why multiple testing correction is essential

Testing 600+ pairs at α = 0.05 would yield ~30 false positives by pure chance
(even if no pair is truly cointegrated). The **Benjamini–Hochberg procedure**
controls the **False Discovery Rate (FDR)** — the expected proportion of false
positives among all rejected hypotheses — at 5%.

This is standard practice in quantitative research and distinguishes the project
from naive academic implementations that ignore this problem.

In [ ]:
print('Running Engle-Granger cointegration tests...')
eg_records = []
for _, row in corr_df.iterrows():
    a, b = row['Stock A'], row['Stock B']
    try:
        _, p, _ = coint(prices[a], prices[b])
        eg_records.append({
            'Stock A': a, 'Stock B': b,
            'Sector A': row['Sector A'], 'Sector B': row['Sector B'],
            'Same Sector': row['Same Sector'],
            'Correlation': row['Correlation'],
            'EG p-value': p
        })
    except Exception:
        pass

eg_df = pd.DataFrame(eg_records)
print(f'EG tests run: {len(eg_df)}')
print(f'Pairs with raw p < 0.05: {(eg_df["EG p-value"] < 0.05).sum()}')

# ── Benjamini-Hochberg FDR correction ──────────────────────────────────────
reject, p_adj, _, _ = multipletests(
    eg_df['EG p-value'].values, alpha=COINT_ALPHA, method='fdr_bh'
)
eg_df['EG p-adj (BH)'] = p_adj.round(6)
eg_df['Pass BH']       = reject

bh_df = eg_df[eg_df['Pass BH']].copy().reset_index(drop=True)
print(f'Pairs surviving Benjamini-Hochberg FDR correction (α=0.05): {len(bh_df)}')
bh_df.sort_values('EG p-adj (BH)').head(15)

## Step 4: Johansen Trace Test — Independent Validation

The Johansen test is the gold standard for multivariate cointegration.
Unlike Engle–Granger (which requires choosing a dependent variable), Johansen
is symmetric and tests the **rank** of the cointegrating space directly.

We use it here as an **independent filter**: a pair must pass both EG (after BH
correction) and Johansen to be considered genuinely cointegrated.

In [ ]:
print('Running Johansen trace tests...')
joh_records = []
for _, row in bh_df.iterrows():
    a, b = row['Stock A'], row['Stock B']
    try:
        sub = prices[[a, b]].dropna()
        res = coint_johansen(sub, det_order=0, k_ar_diff=1)
        # Trace statistic for r=0 hypothesis vs 95% critical value
        trace_stat = res.lr1[0]
        crit_95    = res.cvt[0, 1]
        trace_ratio = trace_stat / crit_95   # >1 means reject r=0
        joh_pass = trace_stat > crit_95
        joh_records.append({
            **row.to_dict(),
            'Johansen Trace': round(trace_stat, 3),
            'Johansen Crit 95%': round(crit_95, 3),
            'Johansen Ratio': round(trace_ratio, 3),
            'Pass Johansen': joh_pass
        })
    except Exception:
        pass

joh_df = pd.DataFrame(joh_records)
both_pass = joh_df[joh_df['Pass Johansen']].copy().reset_index(drop=True)
print(f'Pairs passing both EG(BH) + Johansen: {len(both_pass)}')
both_pass.sort_values('Johansen Ratio', ascending=False)[[
    'Stock A','Stock B','Sector A','Sector B','Correlation',
    'EG p-adj (BH)','Johansen Trace','Johansen Crit 95%','Johansen Ratio'
]].head(20)

## Step 5: Spread Quality Metrics

For each pair surviving cointegration tests, we compute three quality metrics
on the OLS-residual spread:

1. **ADF p-value** — stationarity of the spread (must be < 0.05)
2. **Hurst Exponent** — mean-reversion strength:
   - H < 0.5: mean-reverting (favourable for stat arb)
   - H = 0.5: random walk (no edge)
   - H > 0.5: trending (momentum regime, bad for pairs trading)
3. **Half-Life** — expected days to revert halfway to mean (practical tradability: 5–90 days)

In [ ]:
def hurst_exponent(ts, max_lag=100):
    """R/S Hurst exponent. H<0.5 = mean-reverting."""
    ts = np.array(ts)
    lags = range(2, min(max_lag, len(ts) // 4))
    tau  = [np.sqrt(np.std(ts[lag:] - ts[:-lag])) for lag in lags]
    if any(t == 0 for t in tau):
        return np.nan
    poly = np.polyfit(np.log(lags), np.log(tau), 1)
    return round(poly[0] * 2.0, 4)

def half_life(spread):
    spread = np.array(spread)
    lag    = spread[:-1]
    delta  = np.diff(spread)
    beta   = np.polyfit(lag, delta, 1)[0]
    if beta >= 0:
        return np.inf
    return round(-np.log(2) / beta, 1)

def compute_ols_spread(a_series, b_series):
    model = sm.OLS(a_series, sm.add_constant(b_series)).fit()
    return model.resid, model.params.iloc[1], model.params.iloc[0]

print('Computing spread quality metrics...')
quality_records = []
for _, row in both_pass.iterrows():
    a, b = row['Stock A'], row['Stock B']
    spread, beta, alpha = compute_ols_spread(prices[a], prices[b])
    adf_stat, adf_p, *_ = adfuller(spread.dropna())
    h   = hurst_exponent(spread.dropna())
    hl  = half_life(spread.dropna())
    quality_records.append({
        **row.to_dict(),
        'OLS Beta': round(beta, 4),
        'OLS Alpha': round(alpha, 4),
        'ADF stat': round(adf_stat, 4),
        'ADF p-value': round(adf_p, 6),
        'Hurst': h,
        'Half-life (days)': hl
    })

qual_df = pd.DataFrame(quality_records)
# Apply quality filters
qual_filt = qual_df[
    (qual_df['ADF p-value'] < ADF_ALPHA) &
    (qual_df['Hurst'] < 0.5) &
    (qual_df['Half-life (days)'] >= MIN_HALF_LIFE) &
    (qual_df['Half-life (days)'] <= MAX_HALF_LIFE)
].copy().reset_index(drop=True)

print(f'Pairs after quality filter (ADF + Hurst<0.5 + HL {MIN_HALF_LIFE}-{MAX_HALF_LIFE}d): {len(qual_filt)}')
qual_filt[[
    'Stock A','Stock B','Sector A','Sector B','Correlation',
    'ADF p-value','Hurst','Half-life (days)','Johansen Ratio'
]]

## Step 6: Composite Ranking Score

Each surviving pair is scored on a 0–1 scale across four dimensions
and combined into a weighted composite:

| Dimension | Weight | Direction | Rationale |
|-----------|--------|-----------|----------|
| ADF strength | 30% | lower p → higher score | Stronger stationarity evidence |
| Hurst exponent | 25% | lower H → higher score | Stronger mean reversion |
| Half-life optimality | 20% | closer to 20 days → higher score | Practical trading horizon |
| Johansen ratio | 25% | higher ratio → higher score | Robustness of cointegration |

In [ ]:
def minmax_norm(series, invert=False):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    norm = (series - mn) / (mx - mn)
    return (1 - norm) if invert else norm

df = qual_filt.copy()

# ADF: lower p-value → higher score (invert)
df['score_adf']    = minmax_norm(df['ADF p-value'], invert=True)

# Hurst: lower H → higher score (invert)
df['score_hurst']  = minmax_norm(df['Hurst'], invert=True)

# Half-life: optimal at 20 days, penalise deviation
df['score_hl']     = 1 - minmax_norm((df['Half-life (days)'] - 20).abs())

# Johansen: higher ratio → higher score
df['score_joh']    = minmax_norm(df['Johansen Ratio'])

df['Composite Score'] = (
    0.30 * df['score_adf'] +
    0.25 * df['score_hurst'] +
    0.20 * df['score_hl'] +
    0.25 * df['score_joh']
).round(4)

ranked = df.sort_values('Composite Score', ascending=False).reset_index(drop=True)
ranked.index += 1   # 1-based rank
ranked.index.name = 'Rank'

display_cols = [
    'Stock A','Stock B','Sector A','Sector B',
    'Correlation','ADF p-value','Hurst',
    'Half-life (days)','Johansen Ratio','Composite Score'
]
print(f'\n=== TOP {min(TOP_N_PAIRS, len(ranked))} PAIRS BY COMPOSITE SCORE ===')
ranked[display_cols].head(TOP_N_PAIRS)

In [ ]:
top_n = min(TOP_N_PAIRS, len(ranked))
top   = ranked.head(top_n).copy()
top['Pair'] = top['Stock A'].str.replace('.NS','') + ' / ' + top['Stock B'].str.replace('.NS','')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Top {top_n} Pairs — Quality Metrics', fontsize=14, fontweight='bold')

# Composite score bar chart
axes[0].barh(top['Pair'][::-1], top['Composite Score'][::-1], color='steelblue')
axes[0].set_xlabel('Composite Score')
axes[0].set_title('Composite Ranking')
axes[0].set_xlim(0, 1)

# Hurst exponent
colors = ['green' if h < 0.4 else 'orange' if h < 0.5 else 'red'
          for h in top['Hurst'][::-1]]
axes[1].barh(top['Pair'][::-1], top['Hurst'][::-1], color=colors)
axes[1].axvline(0.5, color='red', linestyle='--', linewidth=1, label='H=0.5 (RW)')
axes[1].set_xlabel('Hurst Exponent')
axes[1].set_title('Mean-Reversion Strength (H < 0.5)')
axes[1].legend(fontsize=8)

# Half-life
axes[2].barh(top['Pair'][::-1], top['Half-life (days)'][::-1], color='darkgreen')
axes[2].axvline(20, color='orange', linestyle='--', linewidth=1, label='Optimal ~20d')
axes[2].set_xlabel('Half-life (trading days)')
axes[2].set_title('Mean-Reversion Speed')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig('top_pairs_quality.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap among stocks in top pairs
top_stocks = list(set(top['Stock A'].tolist() + top['Stock B'].tolist()))
corr_matrix = returns[top_stocks].corr()

labels = [s.replace('.NS','') for s in top_stocks]
fig, ax = plt.subplots(figsize=(max(8, len(top_stocks)), max(6, len(top_stocks)-1)))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
    xticklabels=labels, yticklabels=labels,
    vmin=-0.2, vmax=1.0, center=0.5, ax=ax, linewidths=0.5
)
ax.set_title('Return Correlation — Top Pair Stocks', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('top_pairs_corr_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
save_cols = [
    'Stock A','Stock B','Sector A','Sector B',
    'OLS Beta','OLS Alpha','Correlation',
    'EG p-adj (BH)','Johansen Ratio',
    'ADF p-value','Hurst','Half-life (days)','Composite Score'
]
ranked[save_cols].head(TOP_N_PAIRS).to_csv(TOP_PAIRS_FILE)
print(f'Saved top {TOP_N_PAIRS} pairs to: {TOP_PAIRS_FILE}')
print(f'\nFinal pair list:')
for i, row in ranked.head(TOP_N_PAIRS).iterrows():
    print(f'  {i:2d}. {row["Stock A"]:20s} / {row["Stock B"]:20s}  '
          f'HL={row["Half-life (days)"]:5.1f}d  H={row["Hurst"]:.3f}  '
          f'Score={row["Composite Score"]:.3f}')

## Conclusion

This notebook implements a **production-grade pair discovery pipeline** that differs
from standard academic implementations in three key ways:

1. **Scale:** Tests all 3,081 pairs in the NIFTY 100 universe (vs. a manually chosen pair)
2. **Multiple testing:** Benjamini–Hochberg FDR correction prevents the ~30 false positives
   that a naive 5% threshold would generate at this scale
3. **Dual confirmation:** EG + Johansen combined filter reduces false positives further
4. **Quantified mean reversion:** Hurst exponent directly measures the statistical
   property we are trying to trade — not just whether the pair is stationary,
   but *how strongly* it mean-reverts

The composite-scored top pairs are the input to:
- **NB09** — Walk-forward validation (does performance persist out-of-sample?)
- **NB10** — Multi-pair portfolio construction (risk-diversified deployment)